# connections-rl — 7B scale ablation (QLoRA SFT + GRPO) on Kaggle T4

Does the 1.5B memorization finding hold at 7B? Same data, same reward,
same eval — only the base model changes (Qwen2.5-7B-Instruct, 4-bit).

Settings → Accelerator → **GPU T4 x2**, Internet → On. Run via **Save & Run All (batch)**.

**Resumable by design:** GRPO syncs every checkpoint (every 50 steps) to a
private Hub repo. If the ~12h batch limit kills the run, just Save & Run All
again — the SFT stage is skipped (adapter already on the Hub) and GRPO
resumes from the latest Hub checkpoint.

In [ ]:
# Cell 1 — secrets + GPU check. Use Kaggle Add-ons > Secrets (HF_TOKEN needs write scope).
import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')
HF_USER = 'jacksonlukas'
!nvidia-smi

In [ ]:
# Cell 2 — clone + install + data
!git clone https://github.com/jacksonmlukas/connections-rl.git
%cd connections-rl
!pip install -q -e ".[train]" bitsandbytes
!pip uninstall -q -y torchao
!git clone --depth 1 https://github.com/jacksonmlukas/gvc-local.git /kaggle/working/gvc-local
os.environ['CONNECTIONS_PUZZLES'] = '/kaggle/working/gvc-local/data/puzzles/tagged_connections.json'
!python -m connections_rl.data.build --out data/splits

In [ ]:
# Cell 3 — SFT warm start (skipped if the 7B SFT adapter is already on the Hub)
from huggingface_hub import HfApi, snapshot_download
api = HfApi()
sft_repo = f'{HF_USER}/connections-rl-sft-7b'
if api.repo_exists(sft_repo):
    print('SFT adapter already on Hub — downloading')
    snapshot_download(sft_repo, local_dir='artifacts/sft-7b')
else:
    !python -m connections_rl.train.sft --config configs/train/sft-7b.yaml
    api.create_repo(sft_repo, exist_ok=True)
    api.upload_folder(folder_path='artifacts/sft-7b', repo_id=sft_repo,
                      ignore_patterns=['checkpoint-*'])
    print('pushed', sft_repo)

In [ ]:
# Cell 4 — GRPO (auto-resumes from Hub checkpoints via ckpt_hub_repo)
!python -m connections_rl.train.grpo --config configs/train/grpo-7b.yaml

In [ ]:
# Cell 5 — push the final GRPO adapter
grpo_repo = f'{HF_USER}/connections-rl-grpo-7b'
api.create_repo(grpo_repo, exist_ok=True)
api.upload_folder(folder_path='artifacts/grpo-7b', repo_id=grpo_repo,
                  ignore_patterns=['checkpoint-*'])
print('pushed', grpo_repo)